# LatticeMemory Phase A Training — Colab

**Phase A**: contrastive + hard-negative fine-tuning, **no address loss** (`lambda_address=0`).

Goal: restore float32 geometry (recall@1 toward 0.90) while measuring the natural
query→positive Hamming distance that semantic fine-tuning produces.

**Gate for Phase B**: if natural Hamming after this run is <10 blocks, Phase B
(freeze doc encoder, apply address loss to query tower) is viable.  
If Hamming stays >30, the query-passage gap is fundamental at 128-block resolution.

---
**Runtime**: GPU required. Go to Runtime → Change runtime type → GPU (T4 free or A100 Colab Pro).

## 0. Config — fill these in before running

In [ ]:
# ── USER CONFIG ──────────────────────────────────────────────────────────────

# Where to save results and per-epoch checkpoints on your Drive
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/latticememory/phase_a_1k_5ep"

# GitHub URL for your latticememory repo (or leave REPO_URL=None and
# manually upload latticememory/ to Drive and set PACKAGE_DIR below)
REPO_URL = None  # e.g. "https://github.com/yourname/latticememory.git"
PACKAGE_DIR = None  # e.g. "/content/drive/MyDrive/latticememory_src" if no git

# Training hyperparameters — Phase A
MODEL        = "dfrokido/bge-large-e8-snap"
TRAIN_LIMIT  = 1000   # more data than local runs to improve float32 quality
EVAL_LIMIT   = 100
EPOCHS       = 5
BATCH_SIZE   = 4
GRAD_ACCUM   = 4      # effective batch = 16
LR           = 2e-5
LAMBDA_ADDR  = 0.0    # Phase A: NO address loss
LAMBDA_HARD  = 2.0    # MS MARCO hard negatives on
LAMBDA_NEIGH = 0.5
LOG_EVERY    = 10     # print progress every N batches
# ─────────────────────────────────────────────────────────────────────────────

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted.")

## 2. Install dependencies

In [ ]:
%%capture
!pip install -q sentence-transformers datasets huggingface_hub

## 3. Install latticememory package

In [ ]:
import subprocess, sys, os

if REPO_URL:
    subprocess.run(["git", "clone", REPO_URL, "/content/latticememory"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "/content/latticememory"], check=True)
    print("Installed from GitHub:", REPO_URL)
elif PACKAGE_DIR:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", PACKAGE_DIR], check=True)
    print("Installed from Drive:", PACKAGE_DIR)
else:
    raise RuntimeError(
        "Set REPO_URL (GitHub) or PACKAGE_DIR (Drive path) in the config cell."
    )

import importlib, latticememory
importlib.reload(latticememory)
print("latticememory version:", getattr(latticememory, '__version__', 'dev'))

## 4. Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected — change runtime to GPU."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 5. Pre-run baseline: float32 recall with the base model

This is what Phase A should recover toward.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from latticememory.training import load_msmarco_examples

print("Loading eval examples...", flush=True)
eval_examples = load_msmarco_examples(
    dataset_name="microsoft/ms_marco",
    dataset_config="v1.1",
    split="validation",
    streaming=True,
    min_negatives=1,
    limit=EVAL_LIMIT,
)
print(f"Loaded {len(eval_examples)} eval examples")

def float32_recall(model_path, examples, label=""):
    enc = SentenceTransformer(model_path, device="cuda")
    queries = [e.query for e in examples]
    docs    = [e.positive for e in examples]
    q = enc.encode(queries, normalize_embeddings=True, batch_size=32, show_progress_bar=False)
    d = enc.encode(docs,    normalize_embeddings=True, batch_size=32, show_progress_bar=False)
    sim  = np.dot(q, d.T)
    top1 = np.argmax(sim, axis=1)
    n    = len(examples)
    r    = int(np.sum(top1 == np.arange(n))) / n
    diag = float(np.mean(np.diag(sim)))
    mean = float(np.mean(sim))
    print(f"{label:40s}  recall@1={r:.3f}  diag={diag:.4f}  mean={mean:.4f}")
    return r

base_recall = float32_recall(MODEL, eval_examples, label="Base model")

## 6. Run Phase A training

Saves a checkpoint after every epoch to Drive.  
If the session dies, the latest checkpoint in `DRIVE_OUTPUT_DIR/checkpoints/` is recoverable.

In [ ]:
import json, os
from latticememory.training import train_and_evaluate_msmarco

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f"Output dir: {DRIVE_OUTPUT_DIR}")
print(f"lambda_address={LAMBDA_ADDR}  (Phase A: contrastive+hard only)")
print(f"train_limit={TRAIN_LIMIT}  eval_limit={EVAL_LIMIT}  epochs={EPOCHS}")
print()

result = train_and_evaluate_msmarco(
    training_mode="full_encoder",
    model=MODEL,
    train_limit=TRAIN_LIMIT,
    eval_limit=EVAL_LIMIT,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    lr=LR,
    lambda_address=LAMBDA_ADDR,
    lambda_hard=LAMBDA_HARD,
    lambda_neighborhood=LAMBDA_NEIGH,
    fp16=True,
    gradient_checkpointing=True,
    log_every_batches=LOG_EVERY,
    sts_eval_each_epoch=True,
    sts_source="builtin",
    output_dir=DRIVE_OUTPUT_DIR,
    checkpoint_every_epoch=True,
    device="cuda",
)

print("\n=== Training complete ===")
print(json.dumps({
    "train_mean_hamming_history": result.get("train_mean_hamming_history"),
    "train_min_hamming_history":  result.get("train_min_hamming_history"),
    "contrastive_loss_history":   result.get("contrastive_loss_history"),
    "address_loss_history":       result.get("address_loss_history"),
    "train_lattice_route_rate":   result.get("train", {}).get("lattice_route_rate"),
    "eval_lattice_route_rate":    result.get("eval",  {}).get("lattice_route_rate"),
    "eval_recall_at_1":           result.get("eval",  {}).get("recall_at_1"),
}, indent=2))

## 7. Post-training float32 recall diagnostic

Compare fine-tuned vs base.  
**Target**: fine-tuned recall should recover toward `base_recall`.

In [ ]:
import os
fine_tuned_path = os.path.join(DRIVE_OUTPUT_DIR, "full_encoder")

print(f"Base model recall@1:       {base_recall:.3f}")
ft_recall = float32_recall(fine_tuned_path, eval_examples, label="Phase A fine-tuned")

print()
if ft_recall >= base_recall * 0.95:
    print("PASS: float32 recall preserved. Phase A worked.")
elif ft_recall >= base_recall * 0.85:
    print("PARTIAL: some degradation. Check contrastive_loss_history trend.")
else:
    print("FAIL: recall dropped significantly. Increase train_limit or reduce lr.")

## 8. Hamming distance summary — Phase B gate

This tells you whether fixing the moving-target problem (Phase B) can close the gap.

In [ ]:
print("=== Hamming distance history (query→positive, per epoch) ===")
history = result.get("train_mean_hamming_history", [])
min_h   = result.get("train_min_hamming_history", [])
for i, (mean, mn) in enumerate(zip(history, min_h), 1):
    print(f"  Epoch {i}: mean={mean:.2f}  min={mn}")

final_mean = history[-1] if history else None
print()
if final_mean is not None:
    if final_mean < 10:
        print(f"Phase B VIABLE: mean Hamming={final_mean:.2f} < 10 blocks.")
        print("   → freeze doc encoder from this checkpoint, apply address loss to query tower")
    elif final_mean < 30:
        print(f"Phase B UNCERTAIN: mean Hamming={final_mean:.2f}. More epochs may help.")
        print("   → consider running 10 epochs before committing to Phase B")
    else:
        print(f"Phase B DIFFICULT: mean Hamming={final_mean:.2f} still large.")
        print("   → query/passage may not co-locate at 128-block E8 resolution")
        print("   → discuss architecture before building Phase B")

## 9. Save checkpoint list to Drive

In [ ]:
import os, json
ckpt_root = os.path.join(DRIVE_OUTPUT_DIR, "checkpoints")
if os.path.isdir(ckpt_root):
    epochs_saved = sorted(os.listdir(ckpt_root))
    print("Checkpoints saved:")
    for ep in epochs_saved:
        ep_path = os.path.join(ckpt_root, ep)
        files = os.listdir(ep_path) if os.path.isdir(ep_path) else []
        print(f"  {ep_path}  ({len(files)} files)")
    print()
    print("Latest checkpoint (for Phase B):", os.path.join(ckpt_root, epochs_saved[-1]) if epochs_saved else "none")
else:
    print("No checkpoint directory found — checkpoint_every_epoch may not have fired.")

metrics_path = os.path.join(DRIVE_OUTPUT_DIR, "metrics.json")
print(f"\nFull metrics at: {metrics_path}")